In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../../Evaluation/')
from normal_evaluation.drbart_evaluation import *

In [2]:
model_name = 'helpdesk'
log_name = 'train'

with open('../../transformed_event_logs/Helpdesk_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['Case ID'] = test_event_log['Case ID'].astype(str)
test_event_log['case:concept:name'] = test_event_log['Case ID']
test_event_log['time:timestamp_start'] = test_event_log['Complete Timestamp_start']
test_event_log['time:timestamp_complete'] = test_event_log['Complete Timestamp_complete']


known_resources = ['Value 1', 'Value 10', 'Value 11', 'Value 12', 'Value 13', 'Value 14', 'Value 15', 'Value 16', 'Value 17', 'Value 18', 'Value 19', 'Value 2', 'Value 20', 'Value 21', 'Value 22', 'Value 3', 'Value 4', 'Value 5', 'Value 6', 'Value 7', 'Value 8', 'Value 9']
known_activities = ['Assign seriousness', 'Closed', 'Create SW anomaly', 'DUPLICATE', 'INVALID', 'Insert ticket', 'RESOLVED', 'Require upgrade', 'Resolve SW anomaly', 'Resolve ticket', 'Schedule intervention', 'Take in charge ticket', 'VERIFIED', 'Wait']

In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

likelihoods_A = None
likelihoods_R = None
likelihoods_R_A = None
likelihoods_R_A_S = None
likelihoods_R_A_S_AC = None
likelihoods_R_A_S_RC = None
likelihoods_R_A_S_RC_AC = None
likelihoods_R_A_S_RC_AC_V = None
likelihoods_R_A_S_D = None
likelihoods_R_A_S_D_RC_AC = None

In [4]:
drbart_model_A = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name/',
                     strict_parser=False)
evaluator_A = conduct_evaluation.ConductEvaluation(drbart_model_A, SampleOutcomes_DRBART_Normal_A, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.096382977981501903311737290')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(719266.8398403988)

In [7]:
drbart_model_R_A = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource/',
                     strict_parser=False)
evaluator_R_A = conduct_evaluation.ConductEvaluation(drbart_model_R_A, SampleOutcomes_DRBART_Normal_R_A,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A = evaluator_R_A.sample_cases(False, True)

In [8]:
np.mean([v.ln() for v in likelihoods_R_A[0].values()])

Decimal('-4.093504019849048723657790579')

In [9]:
np.mean(get_pscores(likelihoods_R_A))

np.float64(689384.487185684)

In [10]:
drbart_model_R = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/resource/',
                     strict_parser=False)
evaluator_R = conduct_evaluation.ConductEvaluation(drbart_model_R, SampleOutcomes_DRBART_Normal_R,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R = evaluator_R.sample_cases(False, True)

ValueError: invalid literal for int() with base 10: ''

In [11]:
np.mean([v.ln() for v in likelihoods_R[0].values()])

TypeError: 'NoneType' object is not subscriptable

In [12]:
np.mean(get_pscores(likelihoods_R))

TypeError: 'NoneType' object is not subscriptable

In [13]:
drbart_model_R_A_S = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day/',
                     strict_parser=False)
evaluator_R_A_S = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S, SampleOutcomes_DRBART_Normal_R_A_S,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S = evaluator_R_A_S.sample_cases(False, True)

In [14]:
np.mean([v.ln() for v in likelihoods_R_A_S[0].values()])

Decimal('-4.174919908805762051250948091')

In [15]:
np.mean(get_pscores(likelihoods_R_A_S))

np.float64(748165.6728472952)

In [16]:
drbart_model_R_A_S_AC = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/',
                     strict_parser=False)
evaluator_R_A_S_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_AC, SampleOutcomes_DRBART_Normal_R_A_S_AC,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_AC = evaluator_R_A_S_AC.sample_cases(False, True)

In [17]:
np.mean([v.ln() for v in likelihoods_R_A_S_AC[0].values()])

Decimal('-4.027527058100262006541090489')

In [18]:
np.mean(get_pscores(likelihoods_R_A_S_AC))

np.float64(609562.3409939696)

In [19]:
drbart_model_R_A_S_RC = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/',
                     strict_parser=False)
evaluator_R_A_S_RC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC, SampleOutcomes_DRBART_Normal_R_A_S_RC,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'known_resources' : known_resources
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_RC = evaluator_R_A_S_RC.sample_cases(False, True)

In [20]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC[0].values()])

Decimal('-4.369849728932468458881556187')

In [21]:
np.mean(get_pscores(likelihoods_R_A_S_RC))

np.float64(750639.2113305632)

In [22]:
drbart_model_R_A_S_RC_AC = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count/',
                     strict_parser=False)
evaluator_R_A_S_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_RC_AC,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_RC_AC = evaluator_R_A_S_RC_AC.sample_cases(False, True)

In [23]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC_AC[0].values()])

Decimal('-5.656888343487969282551833505')

In [24]:
np.mean(get_pscores(likelihoods_R_A_S_RC_AC))

np.float64(799298.6222855962)

In [25]:
drbart_model_R_A_S_D = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/',
                     strict_parser=False)
evaluator_R_A_S_D = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D, SampleOutcomes_DRBART_Normal_R_A_S_D,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_D = evaluator_R_A_S_D.sample_cases(False, True)

In [26]:
np.mean([v.ln() for v in likelihoods_R_A_S_D[0].values()])

Decimal('-4.181444040508029666517530221')

In [27]:
np.mean(get_pscores(likelihoods_R_A_S_D))

np.float64(693102.9990769804)

In [28]:
drbart_model_R_A_S_D_RC_AC = DRBART(parser_dir = '../../../../models/standard/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/',
                     strict_parser=False)
evaluator_R_A_S_D_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_D_RC_AC,
                                                   {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n=N)
likelihoods_R_A_S_D_RC_AC = evaluator_R_A_S_D_RC_AC.sample_cases(False, True)

In [29]:
np.mean([v.ln() for v in likelihoods_R_A_S_D_RC_AC[0].values()])

Decimal('-4.817068441908648998919949904')

In [30]:
np.mean(get_pscores(likelihoods_R_A_S_D_RC_AC))

np.float64(841288.8283116684)

In [31]:
results = {
    'drbart_model_A' : likelihoods_A,
    'drbart_model_R' : likelihoods_R,
    'drbart_model_R_A' : likelihoods_R_A,
    'drbart_model_R_A_S' : likelihoods_R_A_S,
    'drbart_model_R_A_S_AC' : likelihoods_R_A_S_AC,
    'drbart_model_R_A_S_RC' : likelihoods_R_A_S_RC,
    'drbart_model_R_A_S_RC_AC' : likelihoods_R_A_S_RC_AC,
    'drbart_model_R_A_S_RC_AC_V' : likelihoods_R_A_S_RC_AC_V,
    'drbart_model_R_A_S_D' : likelihoods_R_A_S_D,
    'drbart_model_R_A_S_D_RC_CC' : likelihoods_R_A_S_D_RC_AC
}
with open('./'+model_name+'_dr_bart_evaluation_'+log_name+'.pickle', 'wb') as handle:
    pickle.dump(results, handle)